In [ ]:
short_names = {
    'Pòlissa/Póliza/Policy': 'POLICY',
    'Tecnologia/Tecnología/Technology': 'TECHNOLOGY',
    'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)': 'DIAMETER',
    'Ús/Uso/Use': 'USAGE',
    "Tipus d'habitatge/Tipo de vivienda/Type of housing": 'HOUSING',
    'Data/Fecha/Date': 'DATETIME',
    'Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)': 'CONSUMPTION',
}

In [ ]:
import pyarrow.dataset as ds
import pandas as pd

file_path = '../data/lectures_horaries_ABD.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()
df = table.to_pandas()
df = df.rename(columns=short_names)
df.drop(columns=['TECHNOLOGY', 'HOUSING'])

# Calculate the differential for each identifier separately
df['FLOW'] = df.groupby('POLICY')['CONSUMPTION'].diff()
df = df.dropna()

# Convert 'Data/Fecha/Date' column to datetime format
df['DATETIME'] = pd.to_datetime(df['DATETIME'])

# Extract the date and hour separately
df['DATE'] = df['DATETIME'].dt.date
df['HOUR'] = df['DATETIME'].dt.hour
df.drop(columns=['DATETIME'])

**Dataset Ajustement (a column per hour for every date and policy)**

In [ ]:
# Pivot the table so that each hour is a separate column
df_pivot = df.pivot_table(index='DATE', columns='HOUR', values='FLOW')

# Rename columns as HOUR0, HOUR1, ..., HOUR23
df_pivot.columns = [f'HOUR{int(hour)}' for hour in df_pivot.columns]

# Reset index to make DATE a column
df_pivot = df_pivot.reset_index()

# Keep only one row per DATE
df_deduped = df.drop_duplicates(subset='DATE')[['DATE', 'POLICY', 'DIAMETER', 'USAGE', 'FLOW']]

# Merge the pivot table with the deduplicated columns
df_final = pd.merge(df_pivot, df_deduped, on='DATE', how='left')

cols = ['POLICY', 'DATE'] + [col for col in df_final.columns if col not in ['POLICY', 'DATE']]
df_final = df_final[cols]

df_final.head()

**Drop of rows with at least one null value**

In [ ]:
hour_columns = [f'HOUR{hour}' for hour in range(24)] # Subset of the hour columns
df_cleaned = df_final.dropna(subset=hour_columns, how='any')

df_cleaned.info()

In [ ]:
# print(df_cleaned.head())  # First 5 rows
df_cleaned.head()

In [ ]:
print(df.columns) # Column names

In [ ]:
print(df.info())  # Data types and missing values 